# EDA + Demo — CIFAR-10 & Model Factory (M1+M2)

Notebook demo cho Practice 2 — chạy được trên CPU, không cần GPU.

**Mục đích:**

- EDA nhanh CIFAR-10 từ `data/processed/*.npy` + `data/splits/*.json` (số liệu thật, không bịa).
- Demo `src.models` factory: dựng 3 backbone với `weights=None` (nhanh, không tải 500MB), verify output `(1, 10)`; số weights thật đối chiếu từ `results/runs/*/model_info.json`.

**Chạy:** mở notebook từ root repo (`e:\lab2`), kernel dùng `.venv` (Python 3.13, torch 2.14.1+cpu).

```powershell
.\.venv\Scripts\python.exe -m notebook
# mở notebooks/20261026_eda_cifar10_demo.ipynb
```

Nguồn số liệu: `results/runs/20261025-150847/metrics.json` (M1), `results/runs/20261026-*/model_info.json` (M2), `data/processed/metadata.json`.

In [ ]:
import json
from pathlib import Path

# Root repo: notebook nằm ở notebooks/ -> parent là root.
# Chạy an toàn dù cwd là root hay notebooks/.
CWD = Path.cwd()
ROOT = CWD if (CWD / "configs").exists() else CWD.parent
print("ROOT:", ROOT.resolve())
assert (ROOT / "data" / "processed" / "metadata.json").exists(), "Không thấy data/processed — chạy `python -m src.data.prepare` trước"

meta = json.loads((ROOT / "data" / "processed" / "metadata.json").read_text(encoding="utf-8"))
split_info = json.loads((ROOT / "data" / "splits" / "cifar10_split_seed42.json").read_text(encoding="utf-8"))
m1 = json.loads((ROOT / "results" / "runs" / "20261025-150847" / "metrics.json").read_text(encoding="utf-8"))
mi_resnet = json.loads((ROOT / "results" / "runs" / "20261026-115633" / "model_info.json").read_text(encoding="utf-8"))
mi_vgg = json.loads((ROOT / "results" / "runs" / "20261026-120218" / "model_info.json").read_text(encoding="utf-8"))
mi_dense = json.loads((ROOT / "results" / "runs" / "20261026-121817" / "model_info.json").read_text(encoding="utf-8"))

print("Classes:", meta["classes"])
print("Split counts:", split_info["counts"])
print("M1 validation:", m1["validation"])
for tag, mi in [("resnet18", mi_resnet), ("vgg11", mi_vgg), ("densenet121", mi_dense)]:
    p = mi["params"]
    print(f"{tag}: total={p['params_total']:,.0f} trainable={p['params_trainable']:,.0f} ({p['params_trainable_pct']:.4f}%) out={mi['output_shape']}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

PROC = ROOT / "data" / "processed"
train_labels = np.load(PROC / "train_labels.npy")  # [50000] int64, tập train gốc
test_labels = np.load(PROC / "test_labels.npy")    # [10000]
classes = meta["classes"]

train_counts = np.bincount(train_labels, minlength=10)
test_counts = np.bincount(test_labels, minlength=10)
print("train_raw per-class:", train_counts.tolist())
print("test per-class:", test_counts.tolist())
assert (train_counts == 5000).all(), "CIFAR-10 train gốc phải 5000/lớp"
assert (test_counts == 1000).all(), "CIFAR-10 test phải 1000/lớp"

# Sau split stratified seed=42: train 4500/lớp, val 500/lớp, test 1000/lớp
x = np.arange(10)
w = 0.25
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(x - w, train_counts, width=w, label="train gốc (5000/lớp)")
ax.bar(x, [4500] * 10, width=w, label="train sau split (4500/lớp)")
ax.bar(x + w, [500] * 10, width=w, label="val (500/lớp)")
ax.set_xticks(x, classes, rotation=20, ha="right")
ax.set_ylabel("Số ảnh")
ax.set_title("Phân bố lớp CIFAR-10 — gốc vs sau split stratified (seed=42)")
ax.legend()
fig.tight_layout()
plt.show()
print("Tổng train+val:", 45000 + 5000, "== 50000:", 45000 + 5000 == 50000)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

PROC = ROOT / "data" / "processed"
# Chỉ đọc ảnh khi cần vẽ (train 146.5MB, test 29.3MB) — uint8 [N,32,32,3]
train_images = np.load(PROC / "train_images.npy", mmap_mode="r")
print("train_images:", train_images.shape, train_images.dtype)

# Thống kê pixel trên 256 ảnh đầu (đã đo ở M1: mean ~119.3, std ~62.9)
sample = np.asarray(train_images[:256]).astype(np.float64)
print(f"mean={sample.mean():.1f} std={sample.std():.1f} "
      f"pct_black={(sample == 0).mean() * 100:.2f}% pct_white={(sample == 255).mean() * 100:.2f}%")

# 1 ảnh đại diện mỗi lớp (ảnh gốc 32x32, chưa resize 224 — resize nằm ở transform lúc train)
train_labels = np.load(PROC / "train_labels.npy")
fig, axes = plt.subplots(2, 5, figsize=(10, 4.5))
for cls in range(10):
    idx = int(np.where(train_labels == cls)[0][0])
    ax = axes[cls // 5, cls % 5]
    ax.imshow(np.asarray(train_images[idx]))
    ax.set_title(f"{cls}: {classes[cls]}")
    ax.axis("off")
fig.suptitle("Mẫu CIFAR-10 — 1 ảnh/lớp (32×32 gốc, hiển thị phóng to)")
fig.tight_layout()
plt.show()

## Demo Model Factory (M2)

- Dựng 3 backbone với `weights=None` để demo nhanh (không tải weights 500MB, mỗi model < 3s).
- Số liệu weights **thật** (ImageNet) lấy từ `results/runs/*/model_info.json` ở cell đầu — không chạy lại download trong notebook.
- Kiểm chứng: `replace_head` (VGG `classifier` là `Sequential[7]`, không phải `Linear`), `freeze_backbone` (< 1% trainable), forward `(1, 3, 224, 224)` → `(1, 10)`.

In [ ]:
import sys
sys.path.insert(0, str(ROOT))  # để `import src.models` khi chạy từ notebooks/
import torch
from src.models.factory import ModelConfig, build_model, count_parameters

REAL = {
    "resnet18": (11181642, 5130, 0.0459),
    "vgg11": (128807306, 40970, 0.0318),
    "densenet121": (6964106, 10250, 0.1472),
}
print(f"{'model':<12}{'total (demo)':>14}{'trainable':>12}{'%':>8}  | total (ImageNet thật)")
for name in ("resnet18", "vgg11", "densenet121"):
    m = build_model(ModelConfig(name=name, weights=None, strategy="freeze_backbone"))
    s = count_parameters(m)
    m.eval()
    with torch.no_grad():
        out = m(torch.randn(1, 3, 224, 224))
    assert tuple(out.shape) == (1, 10), out.shape
    rt, rh, rp = REAL[name]
    print(f"{name:<12}{s['params_total']:>14,.0f}{s['params_trainable']:>12,.0f}{s['params_trainable_pct']:>8.4f}% | {rt:>14,} / {rh:,} ({rp}%) out={tuple(out.shape)}")
    # Cấu trúc head khác nhau — đây là điểm dễ crash nếu code giả định mọi model có `.fc`
    if name == "resnet18":
        print("   head: fc =", type(m.fc).__name__, tuple(m.fc.weight.shape))
    else:
        print("   head: classifier =", type(m.classifier).__name__, len(m.classifier) if hasattr(m.classifier, "__len__") else tuple(m.classifier.weight.shape))
print("OK — cả 3 model forward (1,3,224,224) -> (1,10), freeze < 1% trainable.")

## Kết luận

- Data (M1): 45.000 train / 5.000 val / 10.000 test, stratified seed=42, batch `[64, 3, 224, 224]` float32 — `validation.status = PASS`.
- Model (M2): factory xử lý đúng 2 dạng head (`Linear` vs `Sequential[7]` của VGG11); freeze chỉ giữ head (< 1% params); cả 3 backbone forward đúng `(1, 10)`.
- Chưa train (M3–M5): notebook này không bịa accuracy — số train/val/test acc sẽ có sau khi chạy `src.training` + `src.evaluation`.

Chi tiết: `docs/reports/m2_models_report.md`, `docs/reports/final_report.md`, `docs/AUDIT_LOG.md`.